In [ ]:
# !pip install https://github.com/kyamagu/faiss-wheels/releases/download/v1.7.3/faiss_gpu-1.7.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

### simple faiss gpu

In [1]:
import faiss
import numpy as np
import time

# 1. Setup Parameters
d = 1536  # Vector dimension
nb = 151936  # Database size
nq = 100  # Number of query vectors

# 2. Generate Random Data
print("Generating random data...")
np.random.seed(42)
# Database vectors
xb = np.random.random((nb, d)).astype("float32")
# Query vectors
xq = np.random.random((nq, d)).astype("float32")

# 3. Build the Index on CPU and Move to GPU
print("Building the Faiss index...")

# Step 3a: Create a standard CPU index
cpu_index = faiss.IndexFlatL2(d)

# Step 3b: Create a GPU resource object
# This manages GPU memory and CUDA streams
res = faiss.StandardGpuResources()

# Step 3c: Use the resource object to convert the CPU index to a GPU index
gpu_id = 0  # Use the first GPU
gpu_index = faiss.index_cpu_to_gpu(res, gpu_id, cpu_index)

# 4. Add Vectors to the GPU Index
print(f"Adding {nb} vectors to the GPU index...")
start_time = time.time()
gpu_index.add(xb)
end_time = time.time()

print(f"Vectors added in {end_time - start_time:.2f} seconds.")
print(f"Total vectors in index: {gpu_index.ntotal}")

# 5. Perform the Search
k = 5  # Number of nearest neighbors to find for each query

print(f"\nSearching for the {k} nearest neighbors of {nq} queries...")
start_time = time.time()
distances, indices = gpu_index.search(xq, k)
end_time = time.time()

print(f"GPU search completed in {end_time - start_time:.4f} seconds.")

# 6. Display Results
print("\n--- Search Results ---")
print("Shape of returned indices:", indices.shape)
print("Shape of returned distances:", distances.shape)

for i in range(5):  # Print results for the first 5 queries
    print(f"\nQuery {i}:")
    print(f"  - Nearest neighbor indices: {indices[i]}")
    print(f"  - L2 distances: {distances[i]}")

Generating random data...
Building the Faiss index...
Adding 151936 vectors to the GPU index...
Vectors added in 0.06 seconds.
Total vectors in index: 151936

Searching for the 5 nearest neighbors of 100 queries...
GPU search completed in 0.0027 seconds.

--- Search Results ---
Shape of returned indices: (100, 5)
Shape of returned distances: (100, 5)

Query 0:
  - Nearest neighbor indices: [135698 116457 125758   1979  86973]
  - L2 distances: [222.04825 224.97195 225.40994 225.80557 227.41   ]

Query 1:
  - Nearest neighbor indices: [ 31338  64121 130275  84519 109251]
  - L2 distances: [223.87848 224.63702 225.35855 225.62106 225.91754]

Query 2:
  - Nearest neighbor indices: [ 98052 131774 139670  31560  88476]
  - L2 distances: [226.57025 227.45123 228.79678 229.67374 230.02173]

Query 3:
  - Nearest neighbor indices: [ 18834   5675  57672 145456   9385]
  - L2 distances: [230.09717 230.14667 230.73212 230.7691  230.83765]

Query 4:
  - Nearest neighbor indices: [ 26751  65606 1340

In [1]:
import faiss
import numpy as np
import time

# 1. Setup Parameters
d = 1536  # Vector dimension
nb = 151936  # Database size
nq = 100  # Number of query vectors

# 2. Generate Random Data
print("Generating random data...")
np.random.seed(42)
# Database vectors
xb = np.random.random((nb, d)).astype("float32")
# Query vectors
xq = np.random.random((nq, d)).astype("float32")
# Training vectors (a subset of the database is usually sufficient)
nt = 30000
xt = np.random.random((nt, d)).astype("float32")


# 3. Build the Approximate Index (IVFFlat)
print("Building the Approximate Faiss index...")

nlist = 256  # Number of clusters (cells) to partition the data into.
# A good starting point is round(4 * sqrt(nb))
quantizer = faiss.IndexFlatL2(d)  # The quantizer finds the cluster centroids

# Create the IVF index on the CPU first
cpu_index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)

# Create GPU resources and move the index structure to the GPU
res = faiss.StandardGpuResources()
gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

# 4. Train the Index and Add Vectors
print("Training the index on GPU...")
start_time = time.time()
gpu_index.train(xt)  # Training is done on the GPU
end_time = time.time()
print(f"Training completed in {end_time - start_time:.2f} seconds.")

print(f"Adding {nb} vectors to the GPU index...")
start_time = time.time()
gpu_index.add(xb)  # Adding is also done on the GPU
end_time = time.time()
print(f"Vectors added in {end_time - start_time:.2f} seconds.")
print(f"Total vectors in index: {gpu_index.ntotal}")

# 5. Perform the Search with nprobe
k = 5  # Number of nearest neighbors to find

# Set nprobe: the number of nearby clusters to search.
# This is the key parameter for the speed/accuracy trade-off.
# Higher nprobe = more accurate, but slower.
gpu_index.nprobe = 16

print(f"\nSearching with nprobe = {gpu_index.nprobe}...")
start_time = time.time()
distances, indices = gpu_index.search(xq, k)
end_time = time.time()

print(f"GPU search completed in {end_time - start_time:.4f} seconds.")

# 6. Display Results
print("\n--- Search Results ---")
for i in range(5):
    print(f"\nQuery {i}:")
    print(f"  - Nearest neighbor indices: {indices[i]}")
    print(f"  - L2 distances: {distances[i]}")

Generating random data...
Building the Approximate Faiss index...
Training the index on GPU...
Training completed in 0.19 seconds.
Adding 151936 vectors to the GPU index...
Vectors added in 0.10 seconds.
Total vectors in index: 151936

Searching with nprobe = 16...
GPU search completed in 0.0060 seconds.

--- Search Results ---

Query 0:
  - Nearest neighbor indices: [116457 125758 108597 110517  87331]
  - L2 distances: [224.9718  225.40982 227.46252 229.16734 229.42088]

Query 1:
  - Nearest neighbor indices: [130275  84519  98643  85171  93434]
  - L2 distances: [225.35916 225.62064 227.40454 228.76021 229.69127]

Query 2:
  - Nearest neighbor indices: [ 98052 139670  31560  88476 151791]
  - L2 distances: [226.57063 228.79704 229.67294 230.0223  230.08902]

Query 3:
  - Nearest neighbor indices: [ 9385 59675 40047 27327 12967]
  - L2 distances: [230.83804 231.39282 232.74419 232.83736 233.23164]

Query 4:
  - Nearest neighbor indices: [ 71703  95110 141873  76369  33437]
  - L2 dis

### faiss gpu vs matrix mul

In [1]:
import torch
import faiss
import numpy as np
import time

# --- 1. Setup Parameters and Data ---
d = 1536  # Vector dimension
nb = 151936  # Database size
nq = 1000  # Number of query vectors
k = 10  # Number of nearest neighbors to find

# Use a specific GPU
device = "cuda:0"
print(f"Using device: {device}")
print(f"Faiss has access to {faiss.get_num_gpus()} GPUs.")

# Generate random data
np.random.seed(42)
db_vectors_np = np.random.random((nb, d)).astype("float32")
query_vectors_np = np.random.random((nq, d)).astype("float32")

# --- L2 Normalize the data for Cosine Similarity ---
# Normalization is crucial for comparing dot product with cosine similarity
faiss.normalize_L2(db_vectors_np)
faiss.normalize_L2(query_vectors_np)

# Convert numpy arrays to PyTorch tensors and move to GPU
db_vectors_torch = torch.from_numpy(db_vectors_np).to(device)
query_vectors_torch = torch.from_numpy(query_vectors_np).to(device)


# --- 2. Baseline: PyTorch `matmul` (Brute-Force, 100% Accurate) ---
print("\n--- 1. PyTorch `matmul` Baseline ---")
print("Performing brute-force search with PyTorch...")

start_time = time.time()

# Compute dot products (cosine similarity on normalized vectors)
# (nq, d) @ (d, nb) -> (nq, nb)
similarity_matrix = torch.matmul(query_vectors_torch, db_vectors_torch.T)

# Find the top k most similar vectors
# This gives us the scores (distances) and the indices
baseline_distances, baseline_indices = torch.topk(similarity_matrix, k=k, dim=1)

# Ensure all GPU operations are finished before stopping the timer
torch.cuda.synchronize()
end_time = time.time()

print(f"PyTorch search took: {end_time - start_time:.4f} seconds")
# Move results to CPU for later comparison
baseline_indices = baseline_indices.cpu().numpy()


# --- 3. Faiss Exact Search: `IndexFlatIP` ---
print("\n--- 2. Faiss Exact Search (IndexFlatIP) ---")

# Create an index that uses Inner Product (dot product)
index_flat = faiss.IndexFlatIP(d)

# Move the index to the GPU
res = faiss.StandardGpuResources()
gpu_index_flat = faiss.index_cpu_to_gpu(res, 0, index_flat)

# Add the database vectors
gpu_index_flat.add(db_vectors_np)
print(f"Faiss IndexFlatIP created with {gpu_index_flat.ntotal} vectors.")

# Perform the search
start_time = time.time()
D_flat, I_flat = gpu_index_flat.search(query_vectors_np, k)
end_time = time.time()

print(f"Faiss IndexFlatIP search took: {end_time - start_time:.4f} seconds")


# --- 4. Faiss Approximate Search: `IndexIVFFlat` ---
print("\n--- 3. Faiss Approximate Search (IndexIVFFlat) ---")

nlist = 256  # Number of clusters/cells
quantizer = faiss.IndexFlatIP(d)  # The quantizer also uses Inner Product
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

# Move to GPU
gpu_index_ivf = faiss.index_cpu_to_gpu(res, 0, index_ivf)

# Train the index on the database vectors
print("Training IVF index...")
gpu_index_ivf.train(db_vectors_np)

# Add the vectors
gpu_index_ivf.add(db_vectors_np)
print(f"Faiss IndexIVFFlat created with {gpu_index_ivf.ntotal} vectors.")

# --- Search with different `nprobe` values to see the trade-off ---
for nprobe in [1, 4, 16, 64]:
    gpu_index_ivf.nprobe = nprobe

    start_time = time.time()
    D_ivf, I_ivf = gpu_index_ivf.search(query_vectors_np, k)
    end_time = time.time()

    # Calculate recall@k against the ground truth from PyTorch
    # Recall = (number of true neighbors found) / (total true neighbors)
    # We compare the sets of indices to handle order differences
    found_count = 0
    for i in range(nq):
        true_neighbors = set(baseline_indices[i])
        retrieved_neighbors = set(I_ivf[i])
        found_count += len(true_neighbors.intersection(retrieved_neighbors))

    recall = found_count / (nq * k)

    print(f"\n  nprobe = {nprobe}:")
    print(f"    - Search Time: {end_time - start_time:.4f} seconds")
    print(f"    - Recall@{k}: {recall:.4f}")

Using device: cuda:0
Faiss has access to 1 GPUs.

--- 1. PyTorch `matmul` Baseline ---
Performing brute-force search with PyTorch...
PyTorch search took: 0.0672 seconds

--- 2. Faiss Exact Search (IndexFlatIP) ---
Faiss IndexFlatIP created with 151936 vectors.
Faiss IndexFlatIP search took: 0.0128 seconds

--- 3. Faiss Approximate Search (IndexIVFFlat) ---
Training IVF index...
Faiss IndexIVFFlat created with 151936 vectors.

  nprobe = 1:
    - Search Time: 0.0031 seconds
    - Recall@10: 0.0162

  nprobe = 4:
    - Search Time: 0.0184 seconds
    - Recall@10: 0.0632

  nprobe = 16:
    - Search Time: 0.0984 seconds
    - Recall@10: 0.2145

  nprobe = 64:
    - Search Time: 0.4595 seconds
    - Recall@10: 0.6401


### Text generation

In [1]:
import torch
import faiss
import numpy as np
import time
from transformers import AutoModelForCausalLM, AutoTokenizer


# --- ANSI Color Codes for Terminal Highlighting ---
class Colors:
    GREEN = "\033[92m"  # Green for generated text
    ENDC = "\033[0m"  # Reset to default color


# --- Use torch.no_grad() globally for all inference operations ---
with torch.no_grad():
    # --- 1. Setup Model, Tokenizer, and Faiss Indices ---
    print("--- 1. Initializing Environment ---")
    model_name = "Qwen/Qwen2-1.5B-Instruct"

    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token  # Suppress warning

    device = model.device
    print(f"Model loaded on device: {device} with dtype {model.dtype}")

    # Model parameters
    d = model.config.hidden_size
    max_new_tokens = 30

    # Prompts for warmup and benchmarking
    prompts = [
        "What is the capital of France?",
        "Translate 'hello' to Spanish.",
        "The best way to learn is by",
        "Explain the theory of relativity in one sentence.",
        "A list of common fruits:",
        "Once upon a time, in a land far away,",
        "To be or not to be, that is the",
        "The main components of a computer are",
        "Write a short poem about the ocean.",
        "Photosynthesis is the process by which",
    ]

    # --- Pre-build Faiss Indices ---
    print("\n--- Building Faiss Indices (one-time cost) ---")
    db_np = model.lm_head.weight.to(torch.float32).cpu().numpy()
    res = faiss.StandardGpuResources()
    gpu_index_flat = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatIP(d))
    gpu_index_flat.add(db_np)
    print("Exact Faiss index (IndexFlatIP) is ready.")

    # --- WARMUP PHASE ---
    print("\n--- 2. Starting Warmup Phase ---")
    for p in prompts:
        input_ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        _ = model.generate(input_ids, max_new_tokens=2, do_sample=False)
        outputs = model.model(input_ids, use_cache=False)
        last_token_hidden_state = outputs.last_hidden_state[:, -1, :]
        query_np = last_token_hidden_state.to(torch.float32).cpu().numpy().reshape(1, d)
        _ = gpu_index_flat.search(query_np, k=1)
    torch.cuda.synchronize()
    print("--- Warmup Complete ---\n")

    # Lists to store the full generated texts for comparison
    baseline_full_texts = []
    faiss_full_texts = []

    # --- 3. TIMED BENCHMARK: Standard `model.generate()` ---
    print(
        f"--- 3. TIMED BENCHMARK: Standard `model.generate()` on {len(prompts)} prompts ---"
    )
    torch.cuda.synchronize()
    start_time = time.time()

    for p in prompts:
        input_ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        output_ids = model.generate(
            input_ids, max_new_tokens=max_new_tokens, do_sample=False
        )
        decoded_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        baseline_full_texts.append(decoded_text)

    torch.cuda.synchronize()
    end_time = time.time()
    total_baseline_time = end_time - start_time
    print("--- Baseline Benchmark Complete ---")

    # --- 4. TIMED BENCHMARK: Manual Faiss Method ---
    print(
        f"\n--- 4. TIMED BENCHMARK: Manual Faiss Method on {len(prompts)} prompts ---"
    )
    torch.cuda.synchronize()
    start_time = time.time()

    for p in prompts:
        input_ids_manual = tokenizer(p, return_tensors="pt").input_ids.to(device)
        past_key_values = None
        generated_ids = []

        for i in range(max_new_tokens):
            current_input = (
                input_ids_manual
                if i == 0
                else torch.tensor([[generated_ids[-1]]], device=device)
            )
            outputs = model.model(
                current_input, past_key_values=past_key_values, use_cache=True
            )
            last_token_hidden_state = outputs.last_hidden_state[:, -1, :]
            past_key_values = outputs.past_key_values
            query_np = (
                last_token_hidden_state.to(torch.float32).cpu().numpy().reshape(1, d)
            )
            _, I = gpu_index_flat.search(query_np, k=1)
            next_token_id = I[0][0]
            generated_ids.append(next_token_id)

        full_sequence_ids = input_ids_manual.tolist()[0] + generated_ids
        decoded_text = tokenizer.decode(full_sequence_ids, skip_special_tokens=True)
        faiss_full_texts.append(decoded_text)

    torch.cuda.synchronize()
    end_time = time.time()
    total_faiss_time = end_time - start_time
    print("--- Manual Faiss Benchmark Complete ---")

    # --- 5. Final Performance Results ---
    print("\n\n" + "=" * 50)
    print(" " * 10 + "FINAL PERFORMANCE RESULTS")
    print("=" * 50)
    print(
        f"Benchmark run on {len(prompts)} prompts, generating {max_new_tokens} new tokens each."
    )
    print("-" * 50)
    print(f"Baseline `model.generate()`:")
    print(f"  Total Time: {total_baseline_time:.4f} seconds")
    print(f"  Avg. Time per Prompt: {total_baseline_time / len(prompts):.4f} seconds")
    print("-" * 50)
    print(f"Manual Faiss Method:")
    print(f"  Total Time: {total_faiss_time:.4f} seconds")
    print(f"  Avg. Time per Prompt: {total_faiss_time / len(prompts):.4f} seconds")
    print("-" * 50)

    speed_diff = (total_faiss_time - total_baseline_time) / total_baseline_time * 100
    if speed_diff > 0:
        print(
            f"\nConclusion: The manual Faiss method was {speed_diff:.2f}% SLOWER than the baseline."
        )
    else:
        print(
            f"\nConclusion: The manual Faiss method was {-speed_diff:.2f}% FASTER than the baseline."
        )
    print("=" * 50)

    # --- 6. Qualitative Text Comparison ---
    print("\n\n" + "=" * 50)
    print(" " * 8 + "QUALITATIVE TEXT COMPARISON")
    print("=" * 50)
    divergence_count = 0
    for i in range(len(prompts)):
        prompt_text = prompts[i]
        baseline_full = baseline_full_texts[i]
        faiss_full = faiss_full_texts[i]

        # Isolate the generated part of the text for highlighting
        baseline_gen = (
            baseline_full[len(prompt_text) :]
            if baseline_full.startswith(prompt_text)
            else " [Output Error]"
        )
        faiss_gen = (
            faiss_full[len(prompt_text) :]
            if faiss_full.startswith(prompt_text)
            else " [Output Error]"
        )

        print(f"\n----- Prompt {i+1} -----")
        print(
            f"  Baseline `generate()`: {prompt_text}{Colors.GREEN}{baseline_gen}{Colors.ENDC}"
        )
        print(
            f"  Manual Faiss Method  : {prompt_text}{Colors.GREEN}{faiss_gen}{Colors.ENDC}"
        )

        if baseline_full != faiss_full:
            divergence_count += 1
            print("  *** NOTE: Outputs have diverged! ***")

    print("\n" + "-" * 50)
    print("--- Comparison Summary ---")
    print(
        f"Number of prompts where generated text diverged: {divergence_count} out of {len(prompts)}"
    )
    print("=" * 50)

--- 1. Initializing Environment ---
Model loaded on device: cuda:0 with dtype torch.float16

--- Building Faiss Indices (one-time cost) ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Exact Faiss index (IndexFlatIP) is ready.

--- 2. Starting Warmup Phase ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

--- Warmup Complete ---

--- 3. TIMED BENCHMARK: Standard `model.generate()` on 10 prompts ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

--- Baseline Benchmark Complete ---

--- 4. TIMED BENCHMARK: Manual Faiss Method on 10 prompts ---
--- Manual Faiss Benchmark Complete ---


          FINAL PERFORMANCE RESULTS
Benchmark run on 10 prompts, generating 30 new tokens each.
--------------------------------------------------
Baseline `model.generate()`:
  Total Time: 2.4462 seconds
  Avg. Time per Prompt: 0.2446 seconds
--------------------------------------------------
Manual Faiss Method:
  Total Time: 2.5498 seconds
  Avg. Time per Prompt: 0.2550 seconds
--------------------------------------------------

Conclusion: The manual Faiss method was 4.24% SLOWER than the baseline.


        QUALITATIVE TEXT COMPARISON

----- Prompt 1 -----
  Baseline `generate()`: What is the capital of France? Paris. 

The answer is: Paris. 

Justification: Paris is the capital city of France, as stated in the question and confirmed by common
  Manual Faiss Method  : What is the capital of France? Paris. 

The capital of France is Paris. 

P